# Azure Log Sanitization Notebook (No IP Processing)
This notebook sanitizes **Azure-exported CSV logs** across 7 tables:
- `conn`, `events`, `perf`, `port`, `process`, `security`, `syslog`

Goals:
1. Basic data-quality cleanup (deduplication, timestamp parsing, consistent columns)
2. **Stable pseudonymization** for *non-IP* identifiers (e.g., usernames, hostnames, FQDNs, Azure Resource IDs, GUIDs)
3. Output sanitized datasets to `sanidata/`

### Important constraint
**We do NOT modify, hash, mask, or otherwise process any IP addresses** in this version.
IPs are kept exactly as-is in all columns and in free-text fields.

Recommendation:
- Keep raw inputs in `data/` (private)
- Keep mapping files in `mappings/` and add that folder to `.gitignore`.


In [3]:
# --- 0) Imports & Paths ---
import os, re, json, hashlib
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

BASE_DIR = Path.cwd().parent.resolve().joinpath("data")

# Input files
FILES = {
    "conn":    "azure_conn.csv",
    "events":  "azure_events.csv",
    "perf":    "azure_perf.csv",
    "port":    "azure_port.csv",
    "process": "azure_process.csv",
    "security":"azure_security.csv",
    "syslog":  "syslog.csv",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

INPUTS = collect_inputs(BASE_DIR, FILES)

# Quick sanity print: how many matches per scenario/table
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")

OUT_DIR = BASE_DIR.parent.resolve().joinpath("sanidata")
MAP_DIR = BASE_DIR.parent.resolve().joinpath("mappings") 
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# A stable SALT is used to derive deterministic tokens across runs.
# Keep this secret. Do NOT publish it.
SALT_PATH = MAP_DIR / "salt.txt"
if SALT_PATH.exists():
    SALT = SALT_PATH.read_text().strip()
else:
    SALT = hashlib.sha256(os.urandom(32)).hexdigest()
    SALT_PATH.write_text(SALT)

print("Base data dir:", BASE_DIR.resolve())
print("Output base dir:", OUT_DIR.resolve())
print("Mapping dir:", MAP_DIR.resolve())


sc1: {'conn': 1, 'events': 1, 'perf': 1, 'port': 1, 'process': 1, 'security': 1}
sc2: {'events': 1, 'syslog': 1}
sc3: {'syslog': 1}
sc4: {'syslog': 1}
sc5: {'events': 1}
sc6: {'events': 1, 'syslog': 1}
sc7: {'syslog': 1}
Base data dir: /Users/zhuoran/Projects/SSCMDataset/data
Output base dir: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mapping dir: /Users/zhuoran/Projects/SSCMDataset/mappings


## 1) Stable pseudonymization primitives
We map sensitive identifiers to deterministic tokens (e.g., `USER_001`, `HOST_042`) so that:
- The same real-world entity maps to the same token across all tables.
- The sanitized tables remain joinable for cross-log correlation.

Mapping tables are stored as JSON under `mappings/`.

In [4]:
# --- 1) Mapping helpers (stable across runs) ---

def _load_map(name: str) -> dict:
    p = MAP_DIR / f"{name}.json"
    if p.exists():
        return json.loads(p.read_text())
    return {}

def _save_map(name: str, m: dict):
    p = MAP_DIR / f"{name}.json"
    p.write_text(json.dumps(m, ensure_ascii=False, indent=2))

def stable_hash(text: str) -> str:
    return hashlib.sha256((SALT + str(text)).encode("utf-8")).hexdigest()

def map_token(value, prefix: str, map_name: str, digits: int = 3):
    """Map any value to a stable token PREFIX_XXX, persisted in mappings/<map_name>.json."""
    if pd.isna(value) or value == "":
        return value
    value = str(value)
    m = _load_map(map_name)
    if value in m:
        return m[value]

    # Derive a pseudo-random but deterministic numeric id from the salted hash
    base = int(stable_hash(value)[:8], 16) % (10**digits)
    candidate = f"{prefix}_{base:0{digits}d}"

    # Resolve collisions (rare but possible) deterministically
    used = set(m.values())
    i = 0
    while candidate in used:
        i += 1
        candidate = f"{prefix}_{(base+i)%(10**digits):0{digits}d}"

    m[value] = candidate
    _save_map(map_name, m)
    return candidate


## 2) Regex-based sanitizers (non-IP)
We sanitize the following patterns:
- **Azure Resource IDs**: `/subscriptions/<...>/resourceGroups/<...>/...`
- **GUIDs**: `xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`
- **FQDNs / hostnames in DNS fields**

Again: **IPs are intentionally not sanitized**.

In [5]:
# --- 2) Non-IP regex-based sanitizers ---

GUID_RE = re.compile(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b")
FQDN_RE = re.compile(r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b")
EXCLUDE_TLDS = {"exe", "dll", "sys"}
AZ_RES_RE = re.compile(r"/subscriptions/[^/]+/resourcegroups/[^/]+/providers/[^\s\"]+", re.IGNORECASE)

def _is_empty(x) -> bool:
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == ""

def sanitize_guid(text: str) -> str:
    if _is_empty(text):
        return text

    if pd.isna(text): 
        return text
    s = str(text)
    return GUID_RE.sub(lambda m: map_token(m.group(0).lower(), "GUID", "guid_map", digits=4), s)

def sanitize_resource_id(text: str) -> str:
    """Sanitize Azure Resource ID while preserving structure for analysis."""
    if _is_empty(text):
        return text

    if pd.isna(text):
        return text
    s = str(text)

    def repl(m):
        rid = m.group(0)

        # Replace subscription GUIDs
        rid2 = GUID_RE.sub(lambda mm: map_token(mm.group(0).lower(), "SUB", "sub_map", digits=3), rid)

        # Replace resource group names
        rid2 = re.sub(r"(?i)/resourcegroups/([^/]+)", 
                      lambda mm: f"/resourceGroups/{map_token(mm.group(1), 'RG', 'rg_map', digits=3)}", rid2)

        # Replace VM names
        rid2 = re.sub(r"(?i)/virtualmachines/([^/]+)", 
                      lambda mm: f"/virtualMachines/{map_token(mm.group(1), 'VM', 'vm_map', digits=3)}", rid2)

        return rid2

    return AZ_RES_RE.sub(repl, s)

def sanitize_fqdn(text: str, keep_domain: bool = False) -> str:
    """Sanitize FQDNs. 
    - keep_domain=False: replace the full FQDN with a token + '.example'
    - keep_domain=True: replace only the host label, keep the domain intact
    """
    if _is_empty(text):
        return text
    if pd.isna(text):
        return text
    s = str(text)

    def repl(m):
        fqdn = m.group(0)
        parts = fqdn.split(".")
        tld = parts[-1].lower()
        if tld in EXCLUDE_TLDS:
            return fqdn
        if keep_domain and len(parts) >= 3:
            host = parts[0]
            rest = ".".join(parts[1:])
            return f"{map_token(host, 'HOST', 'host_map', digits=3)}.{rest}"
        return f"{map_token(fqdn.lower(), 'FQDN', 'fqdn_map', digits=4)}.example"

    return FQDN_RE.sub(repl, s)

def sanitize_text_blob(text: str, keep_domain: bool = False) -> str:
    """Apply sanitization to free-text fields (Message, SyslogMessage, etc.).
    Note: This function intentionally does NOT sanitize IP addresses.
    """
    if _is_empty(text):
        return text
    if pd.isna(text):
        return text
    s = str(text)
    s = sanitize_resource_id(s)
    s = sanitize_guid(s)
    s = sanitize_fqdn(s, keep_domain=keep_domain)
    return s


## 3) Load CSVs + basic data-quality cleanup
We parse timestamps (when present) and drop exact duplicates.
Timestamp formats are expected to include day-first formats such as `22/04/2025, 13:03:00.020`.


In [ ]:
# --- 3) Raw load only (NO parsing / NO type conversion / NO dedup) ---

def load_csv_raw(path: Path) -> pd.DataFrame:
    """
    Load CSV without any transformation.
    Keep strings as-is to avoid affecting downstream parsing.
    """
    return pd.read_csv(path, dtype=str, keep_default_na=False)

raw = {}  # raw[scenario][table] = list[DataFrame] OR single DataFrame (your choice)

for sc, tables in INPUTS.items():
    raw[sc] = {}
    for table, paths in tables.items():
        # If you expect exactly one file per (scenario, table), keep it simple:
        if len(paths) == 1:
            raw[sc][table] = load_csv_raw(paths[0])
        else:
            # If there are multiple matches, keep them all (same logic later)
            raw[sc][table] = [load_csv_raw(p) for p in paths]

# Quick overview
summary = {
    sc: {t: (df.shape if not isinstance(df, list) else [d.shape for d in df])
         for t, df in tables.items()}
    for sc, tables in raw.items()
}
summary


## 4) Per-table sanitization rules (Azure)
Design principles:
- Keep semantic/security-relevant fields (event types, ports, protocols, direction, etc.)
- Pseudonymize only identifiers that may leak environment details (usernames, hostnames, FQDNs, GUIDs, Resource IDs)
- Apply regex-based sanitization on free-text columns
- **Do not touch IP addresses**


In [ ]:
# --- 4) Per-table sanitizers ---

def sanitize_common_ids(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Azure / agent identifiers
    for col, pref, mname in [
        ("TenantId", "TEN", "tenant_map"),
        ("AgentId", "AGENT", "agent_map"),
        ("MG", "MG", "mg_map"),
        ("ManagementGroupName", "MGN", "mgn_map"),
    ]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, pref, mname, digits=3))

    # Machine / hostname-like fields
    for col in ["Computer", "HostName", "CollectorHostName", "Machine", "WorkstationName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "HOST", "hostnames_map", digits=3))

    # Azure ResourceId
    if "_ResourceId" in df.columns:
        df["_ResourceId"] = df["_ResourceId"].apply(sanitize_resource_id)

    return df


def sanitize_conn(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    # DNS arrays stored as strings (e.g., ["asavpn..."])
    for col in ["RemoteDnsQuestions", "RemoteDnsCanonicalNames"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_events(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["UserName", "Account", "SubjectUserName", "TargetUserName", "CallerUserName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))
    if "Message" in df.columns:
        df["Message"] = df["Message"].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    if "RenderedDescription" in df.columns:
        df["RenderedDescription"] = df["RenderedDescription"].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_perf(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["CounterPath"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_port(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    return df


def sanitize_process(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["UserName", "UserDomain"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))

    # These fields may embed paths/usernames/FQDNs/GUIDs/ResourceIds
    for col in ["CommandLine", "ExecutablePath", "ParentProcessName", "ProcessName", "Process"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))

    return df


def sanitize_security(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)

    # Account-like identifiers
    for col in ["Account", "TargetAccount", "SubjectUserName", "TargetUserName", "UserName", "CallerUserName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))

    # Textual descriptions often include GUIDs/ResourceIds/FQDNs
    for col in ["RenderedDescription", "Message", "ProcessName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))

    return df


def sanitize_syslog(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["SyslogMessage", "ProcessName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


SANITIZERS = {
    "conn": sanitize_conn,
    "events": sanitize_events,
    "perf": sanitize_perf,
    "port": sanitize_port,
    "process": sanitize_process,
    "security": sanitize_security,
    "syslog": sanitize_syslog,
}


## 5) Run sanitization + write outputs
Outputs:
- `data/sanitized/azure/<table>_sanitized.parquet`
- `data/sanitized/azure/<table>_sanitized.csv` (optional, for compatibility)


In [ ]:
sanitized = {}

for name, df in raw.items():
    print("Sanitizing:", name, df.shape)
    s = SANITIZERS[name](df)
    sanitized[name] = s

    out_pq = OUT_DIR / f"{name}_sanitized.parquet"
    s.to_parquet(out_pq, index=False)

    out_csv = OUT_DIR / f"{name}_sanitized.csv"
    s.to_csv(out_csv, index=False)

print("Done. Files written to:", OUT_DIR.resolve())
sorted([p.name for p in OUT_DIR.glob("*sanitized.*")])[:10]


## 6) Quick spot-check (raw vs sanitized)
This section samples a few rows to confirm that:
- Non-IP identifiers are tokenized
- IPs remain unchanged


In [ ]:
import numpy as np

def sample_compare(name: str, n: int = 3):
    a = raw[name]
    b = sanitized[name]
    if len(a) == 0:
        print("Empty table:", name)
        return
    idx = np.random.choice(len(a), size=min(n, len(a)), replace=False)
    cols = [c for c in a.columns if c in b.columns]
    display(pd.concat(
        [a.iloc[idx][cols].assign(_version="raw"),
         b.iloc[idx][cols].assign(_version="sanitized")],
        axis=0
    ).sort_values("_version"))

for name in ["conn", "events", "process", "syslog"]:
    print("\n===", name, "===")
    sample_compare(name, n=2)


## 7) Optional leak checks
Search for strings you expect to be removed (e.g., organization domains, old nicknames, raw subscription/resource IDs).
Add them to `needles` and run this scan over the sanitized tables.

Note: IPs are not processed, so avoid using IP needles unless you explicitly want to confirm that they remain present.

In [ ]:
def find_leaks(df: pd.DataFrame, needles):
    hits = []
    for col in df.columns:
        if df[col].dtype == object:
            s = df[col].astype(str)
            for needle in needles:
                mask = s.str.contains(re.escape(needle), na=False)
                if mask.any():
                    hits.append((col, needle, int(mask.sum())))
    return hits

needles = [
    "gla.ac.uk",
    "subscriptions/",
    # "azureofficer",
]

for name, df in sanitized.items():
    hits = find_leaks(df, needles)
    if hits:
        print("\nPotential occurrences in", name)
        for col, needle, cnt in hits[:50]:
            print(f"  col={col} needle={needle} count={cnt}")
